# Chapter 30 — Applications of Generative AI: VAEs, Diffusion Models & LLM Agents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part4_transformers_genai/ch30_applications_of_generative_ai/ch30_notebook.ipynb)

Companion notebook for **Chapter 30** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Generative AI creates new content across text, images, video, speech and code, rather than only analysing data.
- A VAE encodes an input into a distribution, and the reparameterization trick keeps that sampling differentiable.
- The KL term in the VAE loss shapes the latent space so that sampling from it produces realistic outputs.
- Diffusion models systematically destroy structure with noise and then learn to reverse the process step by step.
- Decoding strategy matters: beam search explores several partial sequences and usually beats greedy search.
- Hallucination, bias, privacy and interpretability are genuine risks, especially in healthcare applications.

## Example 1: From Autoencoder to Variational Autoencoder

This contrasts the deterministic AANN, which maps an input to a single bottleneck point, with the VAE, whose encoder outputs a mean and log-variance. It then shows the reparameterization trick, z = mu + sigma*eps, showing that the same input yields different latent samples each time, and computes both parts of the VAE loss including the KL term that pulls the latent distribution towards N(0,I).

In [1]:
# Chapter 30 - Example 1: Autoencoder -> VAE (reparameterization trick + KL)
import numpy as np
np.random.seed(0)
np.set_printoptions(suppress=True)

print("=== From Autoencoder to Variational Autoencoder ===\n")
print("AANN / deterministic autoencoder: encoder maps x to a SINGLE point")
print("in the bottleneck (dimension-reduction) layer.")
print("VAE: encoder maps x to a DISTRIBUTION (mean + std) in latent space,")
print("and the decoder samples from it -> we can GENERATE new data.\n")

d, latent = 8, 3
x = np.round(np.random.randn(d), 2)
print(f"Input image x ({d}-dim): {x}\n")

# ---- Deterministic autoencoder ----
W_enc = np.random.randn(d, latent) * 0.4
W_dec = np.random.randn(latent, d) * 0.4
h = np.tanh(x @ W_enc)
x_rec = h @ W_dec
print("--- Deterministic Autoencoder (AANN) ---")
print(f"  bottleneck code h : {np.round(h, 4)}   (one fixed point)")
print(f"  reconstruction    : {np.round(x_rec, 3)}")
print(f"  reconstruction MSE: {np.mean((x - x_rec)**2):.4f}")
print("  Sampling a random h gives arbitrary output - the latent space")
print("  has no structure, so this model is a poor generator.\n")

# ---- VAE encoder: mean and log-variance ----
print("--- Variational Autoencoder (VAE) ---\n")
W_mu     = np.random.randn(d, latent) * 0.4
W_logvar = np.random.randn(d, latent) * 0.4
mu     = x @ W_mu
logvar = x @ W_logvar
sigma  = np.exp(0.5 * logvar)
print("Step 1 - encoder outputs a distribution per dimension:")
print(f"  mu     = {np.round(mu, 4)}")
print(f"  logvar = {np.round(logvar, 4)}")
print(f"  sigma  = exp(0.5*logvar) = {np.round(sigma, 4)}\n")

# ---- Reparameterization trick ----
print("Step 2 - REPARAMETERIZATION TRICK:  z = mu + sigma * epsilon,  eps~N(0,I)")
print("  (sampling directly from N(mu,sigma) is not differentiable;")
print("   moving the randomness into eps makes backpropagation possible)\n")
for trial in range(3):
    eps = np.random.randn(latent)
    z = mu + sigma * eps
    print(f"  draw {trial+1}: eps = {np.round(eps,3)} -> z = {np.round(z,4)}")
print("  Same input x, different z each time -> the decoder generates variety.\n")

# ---- VAE loss ----
print("Step 3 - VAE loss = reconstruction loss + KL divergence\n")
eps = np.random.randn(latent)
z = mu + sigma * eps
x_hat = z @ W_dec
recon = np.mean((x - x_hat) ** 2)
kl = -0.5 * np.sum(1 + logvar - mu**2 - np.exp(logvar))
beta = 1.0
print(f"  Reconstruction loss (MSE)          : {recon:.4f}")
print(f"  KL( q(z|x) || N(0,I) )             : {kl:.4f}")
print(f"  Total loss = recon + beta*KL       : {recon + beta*kl:.4f}")
print("\n  The KL term pulls the latent distribution towards N(0,I), which")
print("  makes the latent space smooth and continuous - so sampling")
print("  z ~ N(0,I) at test time produces realistic new images.\n")

print("KL for special cases (per dimension):")
for m, lv in [(0.0, 0.0), (2.0, 0.0), (0.0, 2.0), (0.0, -2.0)]:
    k = -0.5 * (1 + lv - m**2 - np.exp(lv))
    note = ("perfect match to N(0,1)" if (m, lv) == (0.0, 0.0)
            else "mean far from 0" if m else "variance mismatch")
    print(f"  mu={m:+.1f}, logvar={lv:+.1f} -> KL = {k:.4f}   ({note})")

print("\n--- Denoising Autoencoder ---")
print("  Trained on (noisy image, clean image) pairs; at test time a noisy")
print("  input produces a denoised output. The decoder learns to GENERATE.")

=== From Autoencoder to Variational Autoencoder ===

AANN / deterministic autoencoder: encoder maps x to a SINGLE point
in the bottleneck (dimension-reduction) layer.
VAE: encoder maps x to a DISTRIBUTION (mean + std) in latent space,
and the decoder samples from it -> we can GENERATE new data.

Input image x (8-dim): [ 1.76  0.4   0.98  2.24  1.87 -0.98  0.95 -0.15]

--- Deterministic Autoencoder (AANN) ---
  bottleneck code h : [-0.912   0.3283  0.9376]   (one fixed point)
  reconstruction    : [-0.419  0.456 -0.433  0.344 -0.707 -0.939 -0.034  0.373]
  reconstruction MSE: 2.2784
  Sampling a random h gives arbitrary output - the latent space
  has no structure, so this model is a poor generator.

--- Variational Autoencoder (VAE) ---

Step 1 - encoder outputs a distribution per dimension:
  mu     = [-1.2316 -2.6813  0.6753]
  logvar = [0.3487 1.3959 1.5307]
  sigma  = exp(0.5*logvar) = [1.1905 2.0096 2.1497]

Step 2 - REPARAMETERIZATION TRICK:  z = mu + sigma * epsilon,  eps~N(0,I)

## Example 2: Diffusion Forward and Reverse Processes

This builds the forward diffusion equation step by step, tracking how correlation with the original data decays until x_T is essentially Gaussian noise. It also shows the closed-form shortcut using alpha_bar, then runs the reverse process with a deliberately imperfect denoiser, so you can watch the error fall from 0.78 to 0.03 as structure is progressively restored.

In [2]:
# Chapter 30 - Example 2: Diffusion model forward & reverse processes
import numpy as np
np.random.seed(1)
np.set_printoptions(suppress=True)

print("=== Diffusion Models: Forward and Reverse Processes ===\n")
print("Forward diffusion : slowly DESTROY structure by adding noise")
print("Reverse diffusion : learn to RESTORE structure, turning noise into data\n")
print("Forward step:")
print("  x_(t+1) = sqrt(1 - beta_t) * x_t + sqrt(beta_t) * eps ,  eps ~ N(0, I)")
print("  As T -> infinity, x_T ~ N(0, I)  (pure noise)\n")

# A tiny 1-D 'image' with clear structure
x0 = np.array([1.0, 0.8, -0.6, -1.0, 0.4])
T = 12
betas = np.linspace(0.05, 0.35, T)      # variance schedule

print(f"Original data x_0 : {np.round(x0, 3)}")
print(f"Noise schedule beta_t from {betas[0]:.2f} to {betas[-1]:.2f} over T={T} steps\n")

# ---- Forward process ----
print("--- Forward Diffusion Process (adding noise step by step) ---\n")
print(f"{'t':>3}  {'beta_t':>7}  {'x_t':<42} {'corr with x_0':>14}")
print("-" * 74)
x = x0.copy()
traj = [x.copy()]
for t in range(T):
    eps = np.random.randn(*x.shape)
    x = np.sqrt(1 - betas[t]) * x + np.sqrt(betas[t]) * eps
    traj.append(x.copy())
    if t % 3 == 0 or t == T-1:
        corr = np.corrcoef(x0, x)[0, 1]
        print(f"{t+1:>3}  {betas[t]:>7.3f}  {str(np.round(x,2)):<42} {corr:>14.4f}")

print(f"\nFinal x_T mean = {traj[-1].mean():+.3f}, std = {traj[-1].std():.3f}")
print("The structure of x_0 is essentially destroyed -> x_T is nearly N(0,I).\n")

# ---- Closed form: jump straight to any t ----
print("--- Closed-form shortcut ---\n")
alphas = 1 - betas
alpha_bar = np.cumprod(alphas)
print("Instead of looping, we can jump directly to step t:")
print("  x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps\n")
print(f"{'t':>3}  {'alpha_bar_t':>12}  {'signal kept':>12}  {'noise added':>12}")
print("-" * 46)
for t in [0, 3, 6, 9, T-1]:
    print(f"{t+1:>3}  {alpha_bar[t]:>12.4f}  {np.sqrt(alpha_bar[t]):>12.4f}  {np.sqrt(1-alpha_bar[t]):>12.4f}")
print("\nSignal decays and noise grows monotonically - that is the whole schedule.")

# ---- Reverse process ----
print("\n--- Reverse Diffusion Process (denoising) ---\n")
print("A neural network (a U-Net with conv blocks, self-attention, GroupNorm,")
print("GELU/SiLU activations) predicts the noise eps at each step, and we")
print("subtract it to step back from x_t to x_(t-1).\n")

def predict_noise(x_t, t, err=0.15):
    """Stand-in for the TRAINED denoising network (imperfect, like the real one)."""
    true_eps = (x_t - np.sqrt(alpha_bar[t]) * x0) / np.sqrt(1 - alpha_bar[t])
    return true_eps + err * np.random.randn(*x_t.shape)   # network is not exact

x = traj[-1].copy()
print(f"Start from pure noise x_T : {np.round(x, 3)}")
print("Stepping backwards, each step removes a little predicted noise:\n")
print(f"{'step t':>7}  {'x_t':<40} {'err vs x_0':>11}")
print("-" * 62)
for t in reversed(range(T)):
    eps_hat = predict_noise(x, t)
    # one reverse step: remove predicted noise, keep the schedule scaling
    x0_hat = (x - np.sqrt(1 - alpha_bar[t]) * eps_hat) / np.sqrt(alpha_bar[t])
    if t > 0:
        x = np.sqrt(alpha_bar[t-1]) * x0_hat + np.sqrt(1 - alpha_bar[t-1]) * np.random.randn(*x.shape) * 0.3
    else:
        x = x0_hat
    if t in (11, 8, 5, 2, 0):
        print(f"{t:>7}  {str(np.round(x,3)):<40} {np.abs(x-x0).mean():>11.4f}")

print(f"\nRecovered x_0             : {np.round(x, 3)}")
print(f"Original  x_0             : {np.round(x0, 3)}")
print(f"Mean absolute error       : {np.abs(x - x0).mean():.4f}")
print("The reverse process progressively recovers the structure. It is not")
print("exact because the denoising network is only an approximation - which")
print("is also why real diffusion models use hundreds of small steps.")

print("\n=== DALL-E 2 text-to-image pipeline ===")
print("  1. CLIP text encoder  : caption -> text representation")
print("  2. Diffusion prior    : text encoding -> CLIP image encoding")
print("  3. Reverse diffusion  : image encoding -> final generated image")

=== Diffusion Models: Forward and Reverse Processes ===

Forward diffusion : slowly DESTROY structure by adding noise
Reverse diffusion : learn to RESTORE structure, turning noise into data

Forward step:
  x_(t+1) = sqrt(1 - beta_t) * x_t + sqrt(beta_t) * eps ,  eps ~ N(0, I)
  As T -> infinity, x_T ~ N(0, I)  (pure noise)

Original data x_0 : [ 1.   0.8 -0.6 -1.   0.4]
Noise schedule beta_t from 0.05 to 0.35 over T=12 steps

--- Forward Diffusion Process (adding noise step by step) ---

  t   beta_t  x_t                                         corr with x_0
--------------------------------------------------------------------------
  1    0.050  [ 1.34  0.64 -0.7  -1.21  0.58]                    0.9854
  4    0.132  [ 0.61  0.29 -1.2  -1.05  0.99]                    0.8740
  7    0.214  [-0.48  0.35 -1.27 -1.1   0.9 ]                    0.6743
 10    0.295  [-0.74 -0.46  0.11  0.7   1.18]                   -0.5317
 12    0.350  [-0.    0.03  0.1   0.47  1.21]                   -0.1340

## Example 3: Greedy Search vs Beam Search Decoding

Decoding strategy is one of the LLM components covered in this chapter. Using a small language model, greedy search takes the single most likely word at each step and gets trapped. Beam search instead keeps the B best partial sequences and ends up with a far higher overall probability. It is a concrete illustration of why beam search is worth its extra cost.

In [3]:
# Chapter 30 - Example 3: Decoding strategies - Greedy Search vs Beam Search
import numpy as np
np.random.seed(0)

print("=== Decoding Strategies in LLMs: Greedy vs Beam Search ===\n")
print("After the model produces probabilities over the vocabulary at each")
print("step, a DECODING STRATEGY chooses the actual output words.\n")

vocab = ["the", "cat", "sat", "mat", "<end>"]
V = len(vocab)

# A toy language model: P(next word | last word)
P = {
 "<start>": [0.60, 0.25, 0.05, 0.05, 0.05],
 "the":     [0.05, 0.45, 0.10, 0.35, 0.05],
 "cat":     [0.05, 0.05, 0.70, 0.10, 0.10],
 "sat":     [0.50, 0.05, 0.05, 0.30, 0.10],
 "mat":     [0.05, 0.05, 0.05, 0.05, 0.80],
 "<end>":   [0.00, 0.00, 0.00, 0.00, 1.00],
}
print("Toy language model P(next | current):\n")
print(f"  {'current':<10}" + "".join(f"{w:>9}" for w in vocab))
for k, row in P.items():
    print(f"  {k:<10}" + "".join(f"{p:>9.2f}" for p in row))

MAX = 5

# ---------- Greedy search ----------
print("\n--- Greedy Search: always take the single most likely next word ---\n")
seq, cur, logp = [], "<start>", 0.0
for step in range(MAX):
    probs = np.array(P[cur])
    i = int(np.argmax(probs))
    logp += np.log(probs[i])
    cur = vocab[i]
    seq.append(cur)
    print(f"  step {step+1}: pick '{cur}' (p={probs[i]:.2f})   sequence so far: {' '.join(seq)}")
    if cur == "<end>":
        break
greedy_seq, greedy_logp = seq[:], logp
print(f"\n  Greedy result : {' '.join(greedy_seq)}")
print(f"  log-probability: {greedy_logp:.4f}   (probability {np.exp(greedy_logp):.5f})")

# ---------- Beam search ----------
print("\n--- Beam Search: keep the B best partial sequences at every step ---\n")
B = 3
beams = [([], "<start>", 0.0)]
for step in range(MAX):
    cands = []
    for seq, cur, lp in beams:
        if seq and seq[-1] == "<end>":
            cands.append((seq, cur, lp)); continue
        for i, p in enumerate(P[cur]):
            if p <= 0: continue
            cands.append((seq + [vocab[i]], vocab[i], lp + np.log(p)))
    cands.sort(key=lambda t: -t[2])
    beams = cands[:B]
    print(f"  After step {step+1}, top {B} beams:")
    for s, _, lp in beams:
        print(f"     {' '.join(s):<28} logP = {lp:+.4f}")
    if all(s and s[-1] == "<end>" for s, _, _ in beams):
        break

best_seq, _, best_lp = beams[0]
print(f"\n  Beam search result : {' '.join(best_seq)}")
print(f"  log-probability    : {best_lp:.4f}   (probability {np.exp(best_lp):.5f})")

# ---------- Comparison ----------
print("\n=== Comparison ===\n")
print(f"  {'Strategy':<16}{'Sequence':<30}{'logP':>10}")
print("  " + "-" * 56)
print(f"  {'Greedy':<16}{' '.join(greedy_seq):<30}{greedy_logp:>10.4f}")
print(f"  {'Beam (B=3)':<16}{' '.join(best_seq):<30}{best_lp:>10.4f}")
better = "beam search" if best_lp > greedy_logp else "both equal"
print(f"\n  Higher log-probability found by: {better}")
print("\n  Greedy is fast but can be trapped by a locally attractive word.")
print("  Beam search explores B alternatives and usually finds a better")
print("  overall sequence, at B times the computation cost.")

=== Decoding Strategies in LLMs: Greedy vs Beam Search ===

After the model produces probabilities over the vocabulary at each
step, a DECODING STRATEGY chooses the actual output words.

Toy language model P(next | current):

  current         the      cat      sat      mat    <end>
  <start>        0.60     0.25     0.05     0.05     0.05
  the            0.05     0.45     0.10     0.35     0.05
  cat            0.05     0.05     0.70     0.10     0.10
  sat            0.50     0.05     0.05     0.30     0.10
  mat            0.05     0.05     0.05     0.05     0.80
  <end>          0.00     0.00     0.00     0.00     1.00

--- Greedy Search: always take the single most likely next word ---

  step 1: pick 'the' (p=0.60)   sequence so far: the
  step 2: pick 'cat' (p=0.45)   sequence so far: the cat
  step 3: pick 'sat' (p=0.70)   sequence so far: the cat sat
  step 4: pick 'the' (p=0.50)   sequence so far: the cat sat the
  step 5: pick 'cat' (p=0.45)   sequence so far: the cat sat t